<a href="https://colab.research.google.com/github/Slipstream45/Ignis/blob/master/2DVectorisedNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def euler(x):
  if x < -100:
    return 0.0
  is_negative = False
  if x<0:
    is_negative = True
    x = -x
  t = 1
  n = 50
  termi = 1.0
  for i in range(1,n):
    termi *= (x/i)
    t += termi
  #we did e^x so invert it to get e^-x
  if is_negative:
    return 1.0/t
  else:
    return t

In [ ]:
def square_root(n, precision=1e-10):
  if n<0:
    return None
  if n<1:
    l,h= n,1.0
  else:
    l,h = 0.0,n

  while(h-l) > precision:
    mid = (l+h)/2
    if mid*mid<n:
      l=mid
    else:
      h=mid
  return (l+h)/2

In [ ]:
def lnx(x):
  if x<=0:
    return "Undefined"
  u = x-1
  n=50
  t = 0
  for i in range(1, n+1):
    termi = ((-1)**(i-1)*(u**i))/i
    t += termi
  return t

lnx(1.5)

0.4054651081081643

In [ ]:
def sinx(x):
  n = 50
  t = 0
  termi=x
  for i in range(1,n+1):
    termi *= -(x*x)/((2*i)*(2*i+1))
    t += termi
  return t

sinx(0.57079632679)

-0.03049402092598065

In [ ]:
def cosx(x):
  n =50
  t = 1.0
  termi = 1.0
  for i in range(1,n+1):
    termi*=-(x*x)/((2*i-1)*(2*i))
    t += termi
  return t
cosx(0.9)

0.6216099682706645

In [ ]:
'''
flow: lnx, sinx, cosx, e^x functions implemented
LCG generates 2 numbers random, which are fed into Box-Mueller
which uses all the maths functions, and then gets a
Gaussian Random Number, which can then be used in
Xavier and He, Normal Initialization of Weights
'''
class LCG:
  def __init__(self, seed=45):
    self.state = seed
    self.m = 2**32
    self.a = 1103515245
    self.c = 12345

  #X(n+1) = (aX(n) + c)modm
  def next_uniform(self):
    self.state = (self.a * self.state + self.c)%self.m
    return self.state/self.m

def box_mueller(rnjesus):
  u1 = rnjesus.next_uniform()
  #there is 1/2^32 chance we get a 0.0
  while u1 <= 0.0:
    u1 = rnjesus.next_uniform()
  u2 = rnjesus.next_uniform()
  pi = 3.141592653589793 #i am not approximating this! sorry Ramanujan
  k = (-2)*lnx(u1)
  z0 = square_root(k)*cosx(2*pi*u2)
  z1 = square_root(k)*sinx(2*pi*u2)
  return z0 #maybe go for gaussian someother day...

#Xavier for sigmoid, He for ReLU
def Xavier(n_in,n_out, rng):
  u = 2.0/(n_in+n_out)
  sigma = square_root(u)

  X_matrix = []
  for i in range (n_in):
    row = []
    for j in range(n_out):
      weight = box_mueller(rng)*sigma
      row.append(weight)
    X_matrix.append(row)
  return X_matrix

def He(n_in, n_out, rng):
  u = 2.0/n_in
  sigma = square_root(u)
  H_matrix = []
  for i in range (n_in):
    row = []
    for j in range(n_out):
      weight = box_mueller(rng)*sigma
      row.append(weight)
    H_matrix.append(row)
  return H_matrix

In [ ]:
#log-likelihood (BCE Loss)
def bce_loss(A3, y, lnx):
  m = len(y.matrix)
  loss = 0.0
  #need to have epsilon, so->NN should never predict exactly 1 or 0
  epsilon = 1e-15
  for i in range(m):
    y_true = y.matrix[i][0]
    y_pred = A3.matrix[i][0]

    #if 0 round to 0.000000001
    if y_pred < epsilon:
      y_pred = epsilon
    #if 1 round to 0.999999999
    if y_pred > 1.0-epsilon:
      y_pred = 1.0-epsilon


    loss += -(y_true * lnx(y_pred) + (1.0-y_true) * lnx(1.0-y_pred))
  return loss/m

In [ ]:
class Tensors:
  def __init__(self, matrix, _children=(), _op=''):
    self.matrix = matrix
    self._children = set(_children)
    self._op = _op
    rows = len(matrix)
    cols = len(matrix[0])
    self.grad = [[0.0 for _ in range(cols)] for _ in range(rows)]
    self._backward = lambda : None
  def __repr__(self):
    return f"Tensors({self.matrix})"
  #need Transpose, Multiplication, Element wise (Add,Sub,Mul), ColSum

  def shape(self):
    return (len(self.matrix), len(self.matrix[0]))

  def transpose(self):
    rows = len(self.matrix)
    cols = len(self.matrix[0])
    output = [[0 for _ in range(rows)] for _ in range(cols)]
    for i in range(rows):
      for j in range(cols):
        output[j][i] = self.matrix[i][j]
    return Tensors(output, _children=(self,), _op='T')

  def ew_add(self, other):
    rows = len(self.matrix)
    cols = len(self.matrix[0])
    output = [[0 for _ in range(cols)] for _ in range(rows)]
    for i in range(rows):
      for j in range(cols):
        output[i][j] = self.matrix[i][j] + other.matrix[i][j]
    out = Tensors(output, _children=(self, other), _op='+')

    def _backward():
        for i in range(rows):
          for j in range(cols):
            self.grad[i][j] += out.grad[i][j]
            other.grad[i][j] += out.grad[i][j]
    out._backward = _backward
    return out

  def ew_sub(self, other):
    rows = len(self.matrix)
    cols = len(self.matrix[0])
    output = [[0 for _ in range(cols)] for _ in range(rows)]
    for i in range(rows):
      for j in range(cols):
        output[i][j] = self.matrix[i][j] - other.matrix[i][j]
    out = Tensors(output, _children=(self, other), _op='-')
    def _backward():
      for i in range(rows):
        for j in range(cols):
          self.grad[i][j] += out.grad[i][j]
          out.grad[i][j] -= out.grad[i][j]

    out._backward = _backward
    return out

  def ew_mul(self, other):
    rows = len(self.matrix)
    cols = len(self.matrix[0])
    output = [[0 for _ in range(cols)] for _ in range(rows)]
    for i in range(rows):
      for j in range(cols):
        output[i][j] = self.matrix[i][j] * other.matrix[i][j]

    out = Tensors(output, _children=(self, other), _op='e*')

    def _backward():
      for i in range(rows):
        for j in range(cols):
          self.grad[i][j] += other.matrix[i][j]*out.grad[i][j]
          other.grad[i][j] += self.matrix[i][j]*out.grad[i][j]
    out._backward = _backward
    return out

  def col_add(self):
    rows = len(self.matrix)
    cols = len(self.matrix[0])
    output = []
    for j in range(cols):
      x = 0
      for i in range(rows):
        x += self.matrix[i][j]
      output.append(x)
    return out

  def mat_mul(self, other):
    rows1 = len(self.matrix)
    cols1 = len(self.matrix[0])
    rows2 = len(other.matrix)
    cols2 = len(other.matrix[0])
    output = [[0 for _ in range(cols2)] for _ in range(rows1)]

    if(cols1!=rows2):
      raise ValueError("Dimensionality error")
    else:
      for i in range(rows1):
        for j in range(cols2):
          for k in range(cols1):
            output[i][j] += self.matrix[i][k] * other.matrix[k][j]

    out = Tensors(output, _children=(self, other), _op='*')

    def _backward():
      #this gradient is for weights
      #dW = A_transpose * (error)
      a_transpose = [[0 for _ in range(rows1)] for _ in range(cols1)]
      for i in range(rows1):
        for j in range(cols1):
          a_transpose[j][i] = self.matrix[i][j]

      for i in range(cols1):
        for j in range(cols2):
          for k in range(rows1):
            other.grad[i][j] += a_transpose[i][k] * out.grad[k][j]
      #this is for chain rule
      #dA = W_transpose * dZ (cause immediate differential after dW(n+1) is this)
      #dA = dZ * W_tranpose (for dimensional correctness)
      w_transpose = [[0 for _ in range(rows2)] for _ in range(cols2)]
      for i in range(rows2):
        for j in range(cols2):
          w_transpose[j][i] = other.matrix[i][j]

      for i in range(rows1):
        for j in range(cols1):
          for k in range(cols2):
            self.grad[i][j] += out.grad[i][k] * w_transpose[k][j]

    out._backward = _backward
    return out

  def broadcast(self, other):
    xw_rows = len(other.matrix)
    b_cols = len(self.matrix[0])
    output = []
    for i in range(xw_rows):
      output.append(list(self.matrix[0]))
    out = Tensors(output, _children=(self, other), _op='B')

    def _backward():
      for j in range(b_cols):
        col_error_sums = 0.0
        for i in range(xw_rows):
          col_error_sums += out.grad[i][j]
        self.grad[0][j] += col_error_sums

    out._backward = _backward
    return out

  #activation functions, ReLU on 2, sigmoid on the last output

  def relu(self):
    #f(x) = max(0,x) hadamard product
    rows = len(self.matrix)
    cols = len(self.matrix[0])
    output = [[0 for _ in range(cols)] for _ in range(rows)] #A

    for i in range(rows):
      for j in range(cols):
        if(self.matrix[i][j]>0):
            output[i][j] = self.matrix[i][j]

        else:
          output[i][j] = 0
    out = Tensors(output, _children=(self,), _op='ReLU')

    def _backward():
      for i in range(rows):
        for j in range(cols):
          if self.matrix[i][j]>0:
            self.grad[i][j] += 1*out.grad[i][j]
          else:
            self.grad[i][j] += 0*out.grad[i][j]

    out._backward = _backward
    return out

  def sigmoid(self):
    rows = len(self.matrix)
    cols = len(self.matrix[0])
    output = [[0 for _ in range(cols)] for _ in range(rows)] #A
    #f(x) = 1/(1+e^-x)
    for i in range(rows):
      for j in range(cols):
        x = self.matrix[i][j]
        sigmoid = 1/(1+euler(-x))
        output[i][j] = sigmoid

    out = Tensors(output, _children=(self,), _op='Sigmoid')

    def _backward():
      for i in range(rows):
        for j in range(cols):
          x = out.matrix[i][j]
          self.grad[i][j] += (x*(1-x))*out.grad[i][j]

    out._backward = _backward
    return out

  def backprop(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._children:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    for node in reversed(topo):
      node._backward()

In [ ]:
class OptimizerAdam:
  def __init__(self, parameter_vectors, alpha, beta1, beta2):
    self.parameter_vectors = parameter_vectors
    self.alpha = alpha
    self.beta1 = beta1
    self.beta2 = beta2
    self.t = 0
    #allocate shadow buffers for m and v, fill with zeros
    self.m = [] #cause fucking stupid python cant even declare a unknown dimension matrix ffs!!
    self.v = []
    for p in self.parameter_vectors:
      rows = len(p.matrix)
      cols = len(p.matrix[0])

      #fill m and v now
      m_zero = [[0.0 for _ in range(cols)] for _ in range(rows)]
      v_zero = [[0.0 for _ in range(cols)] for _ in range(rows)]
      self.m.append(m_zero)
      self.v.append(v_zero)


  '''
  parameter_vectors have [W1, b1, W2, b2...], so it contains both W and b
  one method is to have separate functions for weights and biases and have %0 for weights and %1 for biases
  but it's inefficient and for complicated models, it will be a nightmare
  so we have to use a different variable to keep track of the index of the matrix we are processing
  to know if it's a weight or a bias
  '''

  def step(self):
    self.t += 1
    for index, p in enumerate(self.parameter_vectors):
      rows = len(p.matrix)
      cols = len(p.matrix[0])
      for i in range(rows):
        for j in range(cols):
          g = p.grad[i][j]
          #for m
          self.m[index][i][j] = self.beta1*self.m[index][i][j] +(1-self.beta1)*(g)

          #for v
          self.v[index][i][j] = self.beta2*self.v[index][i][j] +(1-self.beta2)*(g**2)

          #bias corrections
          m_hat = self.m[index][i][j]/(1-self.beta1**self.t)
          v_hat = self.v[index][i][j]/(1-self.beta2**self.t)

          p.matrix[i][j] = p.matrix[i][j] - (self.alpha*m_hat)/((v_hat**0.5)+1e-8)

  def thanos_grad(self):
    for p in self.parameter_vectors:
      rows = len(p.matrix)
      cols = len(p.matrix[0])
      p.grad = [[0.0 for _ in range(cols)] for _ in range(rows)]

In [ ]:
if __name__ == "__main__":
#features RCS, Heat, Speed, Air Superiority
  X_raw = [
      [1.2, 0.8, 2.0, 1.5], #su57-target
      [0.1, 0.2, 1.8, 2.0], #f22
      [0.1, 0.1, 2.0, 2.3], #f35
      [0.6, 0.3, 2.0, 1.2], #J20
      [0.2, 0.3, 2.1, 1.7], #AMCA
      [0.2, 0.4, 1.9, 1.6]  #KF21
  ]
  y_raw = [[1.0], [0.0], [1.0], [1.0], [1.0], [1.0]]

  X = Tensors(X_raw)
  y = Tensors(y_raw)

  rnjesus = LCG(seed=45)

  W1 = Tensors(He(4, 6, rnjesus))
  b1 = Tensors([[0.0 for _ in range(6)]])

  W2 = Tensors(He(6, 4, rnjesus))
  b2 = Tensors([[0.0 for _ in range(4)]])

  W3 = Tensors(Xavier(4, 1, rnjesus))
  b3 = Tensors([[0.0 for _ in range(1)]])

  epochs = 1000
  alpha = 0.01
  all_params = [W1, b1, W2, b2, W3, b3]
  adam = OptimizerAdam(parameter_vectors=all_params, alpha=0.01, beta1=0.9, beta2=0.999)

  print("beginning training loop->")
  for epoch in range(epochs):

    #Layer 1 (ReLU)
    XW1 = X.mat_mul(W1)
    Z1 = XW1.ew_add(b1.broadcast(XW1))
    A1 = Z1.relu()

    # Layer 2 (ReLU)
    A1W2 = A1.mat_mul(W2)
    Z2 = A1W2.ew_add(b2.broadcast(A1W2))
    A2 = Z2.relu()

    # Layer 3 (Sigmoid)
    A2W3 = A2.mat_mul(W3)
    Z3 = A2W3.ew_add(b3.broadcast(A2W3))
    A3 = Z3.sigmoid()

    loss = bce_loss(A3, y, lnx)
    if epoch%100 ==0:
      print(f"Epoch: {epoch} | Loss: {loss:.6f}")

    for i in range(len(Z3.matrix)):
      Z3.grad[i][0] = A3.matrix[i][0] - y.matrix[i][0]

    Z3.backprop()
    A2.backprop()
    Z2.backprop()
    A1.backprop()
    Z1.backprop()

    adam.step()
    adam.thanos_grad()
print("--- TRAINING COMPLETE ---")
print("Final Target Predictions:")
print("(Target: F-22[1], Su-57[0], F-35[1], J-20[1], AMCA[1], KF21[1])")
for i, row in enumerate(A3.matrix):
    print(f"Jet {i+1}: [{row[0]:.4f}]")

beginning training loop->
Epoch: 0 | Loss: 0.559751
Epoch: 100 | Loss: 0.298306
Epoch: 200 | Loss: 0.203281
Epoch: 300 | Loss: 0.101392
Epoch: 400 | Loss: 0.042140
Epoch: 500 | Loss: 0.021332
Epoch: 600 | Loss: 0.013240
Epoch: 700 | Loss: 0.009090
Epoch: 800 | Loss: 0.006479
Epoch: 900 | Loss: 0.005060
--- TRAINING COMPLETE ---
Final Target Predictions:
(Target: F-22[1], Su-57[0], F-35[1], J-20[1], AMCA[1], KF21[1])
Jet 1: [1.0000]
Jet 2: [0.0208]
Jet 3: [0.9970]
Jet 4: [1.0000]
Jet 5: [1.0000]
Jet 6: [1.0000]
